# E-Commerce Lakehouse: Gold Business Metrics

## Monthly sales for delivered orders

Group delivered orders by purchase month.
Report merchandise sales, freight, recorded payments, and average
merchandise value per order separately.

Amounts are in Brazilian reais (BRL). Sales represent source item
prices for delivered orders, not accounting net revenue or profit.
Incomplete purchase months and undelivered orders affect comparisons.

In [ ]:
from pyspark.sql import functions as F

gold_orders_df = spark.table(
    "workspace.ecommerce.gold_order_summary"
)

delivered_orders_df = gold_orders_df.filter(
    F.col("order_status") == "delivered"
)

monthly_sales_df = (
    delivered_orders_df
    .groupBy("purchase_month")
    .agg(
        F.count("*").alias("delivered_order_count"),
        F.countDistinct("customer_unique_id").alias("customer_count"),
        F.sum("merchandise_value").alias("merchandise_sales_brl"),
        F.sum("freight_value").alias("freight_brl"),
        F.sum("recorded_payment_value").alias("recorded_payments_brl"),
        F.round(
            F.avg("merchandise_value"), 2
        ).alias("avg_merchandise_per_order_brl"),
        F.count(
            F.when(F.col("missing_items_flag"), 1)
        ).alias("orders_missing_items"),
        F.count(
            F.when(F.col("missing_payments_flag"), 1)
        ).alias("orders_missing_payments")
    )
)

(
    monthly_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_monthly_sales")
)

# Reconcile the monthly order counts with the delivered population.
monthly_order_count = monthly_sales_df.agg(
    F.sum("delivered_order_count").alias("orders")
).first()["orders"]

delivered_order_count = delivered_orders_df.count()

print("Delivered source orders:", delivered_order_count)
print("Orders represented in monthly metrics:", monthly_order_count)

if monthly_order_count != delivered_order_count:
    raise ValueError("Monthly order counts do not reconcile.")

display(monthly_sales_df.orderBy("purchase_month"))

## Product category performance

Join delivered order items to Silver products and summarize sales
by the original product category name. Label missing categories
as unknown without excluding their sales.

Count item rows and distinct orders separately. An order containing
multiple categories can contribute to more than one category,
so category order counts must not be added together.

In [ ]:
items_df = spark.table("workspace.ecommerce.silver_order_items")
products_df = spark.table("workspace.ecommerce.silver_products")

delivered_items_df = items_df.join(
    delivered_orders_df.select("order_id"),
    on="order_id",
    how="inner"
)

category_items_df = (
    delivered_items_df
    .join(
        products_df.select("product_id", "product_category_name"),
        on="product_id",
        how="left"
    )
    .withColumn(
        "product_category",
        F.when(
            F.col("product_category_name").isNull()
            | (F.trim(F.col("product_category_name")) == ""),
            F.lit("unknown")
        ).otherwise(F.col("product_category_name"))
    )
)

category_sales_df = (
    category_items_df
    .groupBy("product_category")
    .agg(
        F.count("*").alias("item_count"),
        F.countDistinct("order_id").alias("order_count"),
        F.countDistinct("product_id").alias("product_count"),
        F.sum("price").alias("merchandise_sales_brl"),
        F.sum("freight_value").alias("freight_brl")
    )
)

source_totals = delivered_items_df.agg(
    F.count("*").alias("items"),
    F.sum("price").alias("sales")
).first()

category_totals = category_sales_df.agg(
    F.sum("item_count").alias("items"),
    F.sum("merchandise_sales_brl").alias("sales")
).first()

print("Delivered source items:", source_totals["items"])
print("Items represented in categories:", category_totals["items"])
print("Delivered merchandise sales:", source_totals["sales"])
print("Category merchandise sales:", category_totals["sales"])

if (
    source_totals["items"] != category_totals["items"]
    or source_totals["sales"] != category_totals["sales"]
):
    raise ValueError("Category totals do not reconcile.")

(
    category_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_category_sales")
)

display(
    category_sales_df.orderBy(
        F.col("merchandise_sales_brl").desc()
    ).limit(10)
)

## Customer purchase metrics

Summarize delivered orders by customer_unique_id to recognize
customers across purchases.

Calculate delivered order count, merchandise sales, and first
and latest purchase dates. Repeat customers have at least two
delivered orders in this dataset's observation period.

In [ ]:
customer_metrics_df = (
    delivered_orders_df
    .groupBy("customer_unique_id")
    .agg(
        F.count("*").alias("delivered_order_count"),
        F.sum("merchandise_value").alias("merchandise_sales_brl"),
        F.min("purchase_date").alias("first_purchase_date"),
        F.max("purchase_date").alias("latest_purchase_date")
    )
    .withColumn(
        "repeat_customer_flag",
        F.col("delivered_order_count") >= 2
    )
)

customer_totals = customer_metrics_df.agg(
    F.count("*").alias("customers"),
    F.sum("delivered_order_count").alias("orders"),
    F.sum("merchandise_sales_brl").alias("sales"),
    F.count(
        F.when(F.col("repeat_customer_flag"), 1)
    ).alias("repeat_customers")
).first()

source_totals = delivered_orders_df.agg(
    F.count("*").alias("orders"),
    F.sum("merchandise_value").alias("sales")
).first()

print("Customers with delivered orders:", customer_totals["customers"])
print("Repeat customers:", customer_totals["repeat_customers"])
print("Delivered source orders:", source_totals["orders"])
print("Orders represented in customer metrics:", customer_totals["orders"])
print("Source merchandise sales:", source_totals["sales"])
print("Customer merchandise sales:", customer_totals["sales"])

if (
    source_totals["orders"] != customer_totals["orders"]
    or source_totals["sales"] != customer_totals["sales"]
):
    raise ValueError("Customer metrics do not reconcile.")

(
    customer_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_customer_metrics")
)

display(
    customer_metrics_df.orderBy(
        F.col("merchandise_sales_brl").desc()
    ).limit(10)
)

## Delivery performance by customer state

Summarize delivered orders by customer state. Report average
calendar days from purchase to delivery and carrier transit time.

Calculate late delivery percentage only among orders with known
actual and estimated delivery dates. Exclude flagged shipping
sequences from carrier transit averages, while retaining those
orders in other eligible metrics.

In [ ]:
delivery_metrics_df = (
    delivered_orders_df
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered_order_count"),
        F.count("delivery_days").alias("orders_with_delivery_duration"),
        F.round(
            F.avg("delivery_days"), 2
        ).alias("avg_delivery_days"),
        F.count("carrier_transit_days").alias("orders_with_valid_transit"),
        F.round(
            F.avg("carrier_transit_days"), 2
        ).alias("avg_carrier_transit_days"),
        F.count("late_delivery_flag").alias("orders_with_lateness_check"),
        F.count(
            F.when(F.col("late_delivery_flag"), 1)
        ).alias("late_order_count"),
        F.count(
            F.when(F.col("shipping_sequence_issue"), 1)
        ).alias("shipping_sequence_issue_count")
    )
    .withColumn(
        "late_delivery_pct",
        F.when(
            F.col("orders_with_lateness_check") > 0,
            F.round(
                100.0 * F.col("late_order_count")
                / F.col("orders_with_lateness_check"),
                2
            )
        )
    )
)

summary = delivery_metrics_df.agg(
    F.sum("delivered_order_count").alias("orders"),
    F.sum("orders_with_lateness_check").alias("eligible_orders"),
    F.sum("late_order_count").alias("late_orders"),
    F.sum("shipping_sequence_issue_count").alias("shipping_issues")
).first()

source_order_count = delivered_orders_df.count()

print("Delivered source orders:", source_order_count)
print("Orders represented in delivery metrics:", summary["orders"])
print("Orders eligible for lateness checks:", summary["eligible_orders"])
print("Late orders:", summary["late_orders"])
print("Shipping sequence issues:", summary["shipping_issues"])

if summary["orders"] != source_order_count:
    raise ValueError("Delivery order counts do not reconcile.")

(
    delivery_metrics_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.ecommerce.gold_delivery_metrics")
)

display(
    delivery_metrics_df.orderBy(
        F.col("delivered_order_count").desc()
    )
)